In [1]:
# Standard library imports
import os
import random
import sys

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cProfile
import pstats

# Configure matplotlib
plt.style.use('fig.style')
figsize = (8,4)

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Local imports
from src import get_pairwise_augseqs
from src.get_hg_projection_matrix import get_hg_projection_matrix
from src.matrix_visualizer import visualize_matrix
from src.seq_embedder import SeqEmbedder

In [2]:
L = 40
alphabet = ['A', 'C', 'G', 'T']  
augseqs = get_pairwise_augseqs(seq_length=L, alphabet=alphabet, wildcard_char='*')
N = len(augseqs)
theta = pd.Series(index=augseqs, data=np.random.randn(N))
print(f'{L=}\n{N=:,}\n{alphabet=}\n{theta[:5]=}\n{theta[-5:]=}')

# Needed for profiling
wt_seq = ('ACGT'*L)[:L]
bg_df = pd.DataFrame(index=range(L), columns=alphabet, data={'A': .1, 'C': .4, 'G': .4, 'T': .1})


L=40
N=12,641
alphabet=['A', 'C', 'G', 'T']
theta[:5]=****************************************   -0.252185
A***************************************   -0.456793
C***************************************   -1.551536
G***************************************    1.470341
T***************************************    0.977568
dtype: float64
theta[-5:]=**************************************GT    0.725999
**************************************TA   -0.112015
**************************************TC   -0.483352
**************************************TG    0.144394
**************************************TT    0.678453
dtype: float64


In [3]:
# Get encoder; use to encode sequences
embedder = SeqEmbedder(augseqs=augseqs)

# Get projection matrix
P = get_hg_projection_matrix(augseqs=augseqs, alphabet=alphabet, bg_type='uniform', wildcard_char='*', out_type='sparse')

# Check that gauge matches for every sequence
num_seqs_to_test = 100
for seq_num in range(num_seqs_to_test):
    seq = ''.join(random.choices(alphabet, k=L))

    # Get embedded sequence
    x =embedder.embed(seq)

    # Compute function using the two vectors
    f = theta@x
    f_fixed = (P@theta)@x
    
    # Check that the two functions are close
    assert np.isclose(f, f_fixed), f'{f=}\n{f_fixed=}'
print(f'Tested {num_seqs_to_test} random sequences; all seqs passed.')

# Visualize projection matrix
#visualize_matrix(P.values, show_grid=False, figsize=figsize)

Tested 100 random sequences; all seqs passed.


In [4]:
# Default keyword arguments that are common across all background types
default_kwargs = {
    'augseqs': augseqs,
    'alphabet': alphabet, 
    'wildcard_char': '*',
    'out_type': 'sparse'
}

# Variable keyword arguments for different background types
bg_type_kwargs = {
    'wildtype': {'bg_type': 'wildtype', 'wt_seq': wt_seq},
    'uniform': {'bg_type': 'uniform'},
    'custom': {'bg_type': 'custom', 'bg_df': bg_df},
}

# Update each params dict with the default kwargs
for bg_type in bg_type_kwargs:
    bg_type_kwargs[bg_type].update(default_kwargs)

# Loop over background types
for bg_type, kwargs in bg_type_kwargs.items():
    print(f"\nProfiling with {bg_type} background:")
    with cProfile.Profile() as profiler:
        P_df = get_hg_projection_matrix(**kwargs)
        
    stats = pstats.Stats(profiler)
    stats.strip_dirs().sort_stats('cumulative').print_stats(10)



Profiling with wildtype background:
         442302 function calls (442281 primitive calls) in 1.285 seconds

   Ordered by: cumulative time
   List reduced from 280 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      8/7    0.005    0.001    1.120    0.160 selectors.py:558(select)
        1    0.832    0.832    0.963    0.963 get_hg_projection_matrix.py:8(get_hg_projection_matrix)
      8/7    0.000    0.000    0.163    0.023 events.py:86(_run)
      8/7    0.005    0.001    0.163    0.023 {method 'run' of '_contextvars.Context' objects}
    12641    0.083    0.000    0.163    0.000 get_suborbit_augseqs.py:5(get_suborbit_augseqs)
        4    0.000    0.000    0.151    0.038 zmqstream.py:574(_handle_events)
        2    0.000    0.000    0.151    0.075 asyncio.py:200(_handle_events)
        4    0.000    0.000    0.150    0.038 zmqstream.py:615(_handle_recv)
        4    0.000    0.000    0.150    0.038 zmqstream.py:547(_run_ca